# Ayudantía 07: Threading 🚀

## Ayudantes 👾

Y sus recomendaciones semanales 🎵

- S1: Josefa Buch
  - [Los Fabulosos Cadillacs - Calaveras Y Diablitos](https://www.youtube.com/watch?v=Pev2i1f1DWg&list=RDPev2i1f1DWg&start_radio=1)
- S2: Bastián Pérez
  - [ABBA - The Winner Takes It All](https://www.youtube.com/watch?v=92cwKCU8Z5c&list=RD92cwKCU8Z5c&start_radio=1)
- S3: Daniel Villaseñor
  - [Depeche Mode - Mercy In You](https://www.youtube.com/watch?v=zbLhWOfRgCk&list=RDzbLhWOfRgCk&start_radio=1)
- S4: André Angulo
  - [Calvin Harris - Feel So Close](https://www.youtube.com/watch?v=R8egmh5dWzo&list=RDR8egmh5dWzo&start_radio=1)
- S5: Clemente Campos
  - [Boygenius - Not Strong Enough](https://www.youtube.com/watch?v=rTcF-tZwlXI&list=RDrTcF-tZwlXI&start_radio=1)

Con aparición especial de:

- Pablo Araneda (Profe S2)
  - [Shibayan feat. Hatsune Miku - sugar chocolate waffle](https://youtu.be/rL8iog0cK7E)
- Daniela Concha (Profe S5)
  - [ILOVEFLOWERS! - waltz (tuesday)](https://open.spotify.com/track/5xdr7cDrcAbHjzsyl0zISd)


## Contenidos 📖

- Módulo `threading` en Python
- Creación y ejecución de hilos con `Thread`
- Condiciones de carrera y su corrección con `Lock`
- Coordinación de hilos con `Event`
- Comportamiento de *daemon threads*


## 📋 Tabla Resumen: Threading

A continuación se presenta un resumen de los comandos más importantes del módulo `threading`

### Creación y ejecución de hilos

| Comando | Descripción |
|---|---|
| `threading.Thread(target=f, args=(...))` | Crea un hilo que ejecutará la función `f` con los argumentos dados |
| `threading.Thread(target=f, name='nombre')` | Crea un hilo con nombre identificador |
| `thread.start()` | Inicia la ejecución del hilo (internamente llama al método `run()`) |
| `thread.join(timeout=None)` | El hilo que llama queda bloqueado esperando que `thread` termine (o hasta `timeout` segundos) |
| `thread.is_alive()` | Retorna `True` si el hilo todavía está en ejecución |
| `thread.daemon = True` | Marca el hilo como *daemon*: se detiene automáticamente cuando el programa principal termina |
| `threading.current_thread()` | Retorna una referencia al hilo que está ejecutando el código en ese momento |

### Sincronización con Lock

| Comando | Descripción |
|---|---|
| `lock = threading.Lock()` | Crea un candado para proteger secciones críticas |
| `lock.acquire()` | Adquiere el lock; si ya está tomado, el hilo queda bloqueado esperando |
| `lock.release()` | Libera el lock para que otro hilo pueda adquirirlo |
| `with lock:` | Equivalente a `acquire()` + bloque de código + `release()` |

> ⚠️ El lock que comparten los hilos **debe ser la misma instancia**. Si cada hilo crea su propio lock, no hay protección real.

### Señalización con Event

| Comando | Descripción |
|---|---|
| `evento = threading.Event()` | Crea un objeto de señalización entre hilos (flag interno inicialmente en `False`) |
| `evento.set()` | Activa la señal (flag → `True`). Despierta a todos los hilos que estén esperando con `wait()` |
| `evento.clear()` | Desactiva la señal (flag → `False`) |
| `evento.wait(timeout=None)` | El hilo se pausa hasta que la señal esté activa. Si se pasa `timeout`, espera a lo más ese tiempo y retorna |
| `evento.is_set()` | Retorna `True` si la señal está activa, **sin pausar el hilo** |

### Ejecución diferida con Timer

| Comando | Descripción |
|---|---|
| `timer = threading.Timer(t, f)` | Crea un hilo que ejecutará `f` después de `t` segundos |
| `timer.start()` | Inicia la cuenta regresiva |
| `timer.cancel()` | Cancela el timer antes de que se ejecute |

## Introducción

En esta ayudantía trabajaremos conceptos fundamentales de **concurrencia y threading en Python**. El foco estará en identificar y corregir problemas de sincronización, y en coordinar hilos usando **señales** (`Event`). Practicaremos el diagnóstico de condiciones de carrera, el uso correcto de `Lock`, y la comunicación entre hilos mediante eventos.


## Estación Espacial DCC 🛸

La **Estación Espacial DCC** opera con múltiples sistemas funcionando en paralelo: astronautas que consumen recursos compartidos, módulos de la nave que se preparan de forma independiente y un controlador de misión que coordina el lanzamiento. Tu misión es asegurarte de que todos estos sistemas funcionen correctamente bajo concurrencia.


## Ejercicio 1: Oxígeno en peligro 🧯

La estación espacial tiene un sistema de monitoreo de oxígeno compartido. Cada astronauta consume oxígeno periódicamente. Sin embargo, el sistema actual tiene un bug crítico que provoca lecturas incorrectas del nivel de oxígeno debido a una **condición de carrera**.

Además, la comandancia de la estación ahora exige llevar un registro exacto del total de oxígeno consumido por cada astronauta.

**Tu misión es implementar lo siguiente:**

- **Registro de consumo:** Modifica la clase `EstacionEspacial` agregando un diccionario (`self.registro`) que indique cuanto oxígeno ha consumido cada astronauta.
- **Actualizar consumo:** El método `consumir_oxigeno` recibe el nombre del astronauta además de la cantidad de oxígeno que consumió. Deberá reducir el nivel de oxígeno y actualizar el registro de consumo para el astronauta.

El nivel inicial es **2000 unidades**. Hay **5 astronautas**, consumiendo **distintas unidades** en **50 iteraciones**. Si lo arreglas bien, el nivel final debería ser **350 unidades** y el registro debería tener el total consumido correspondiente para cada uno.


In [ ]:
import threading
import time


class EstacionEspacial:
    def __init__(self, oxigeno_inicial: int) -> None:
        self.oxigeno = oxigeno_inicial
        self.lock = threading.Lock()  # Lock compartido para proteger recursos
        self.registro = {}  # Registro por astronauta

    def consumir_oxigeno(self, astronauta: str, cantidad: int) -> None:
        with self.lock:  # Proteger ambos accesos (oxigeno y diccionario)
            nivel_actual = self.oxigeno
            time.sleep(0.00001)
            self.oxigeno = nivel_actual - cantidad

            if astronauta not in self.registro:
                self.registro[astronauta] = 0
            self.registro[astronauta] += cantidad


class Astronauta(threading.Thread):
    def __init__(self, nombre: str, estacion: EstacionEspacial, consumo: int) -> None:
        super().__init__(name=nombre)
        self.estacion = estacion
        self.consumo = consumo

    def run(self) -> None:
        for _ in range(50):
            self.estacion.consumir_oxigeno(self.name, self.consumo)
            time.sleep(0.001)


if __name__ == '__main__':
    estacion = EstacionEspacial(oxigeno_inicial=2000)
    astronautas = [
        Astronauta('Clemente', estacion, consumo=10),  # Clemente consume mucho oxígeno
        Astronauta('Bastián', estacion, consumo=6),
        Astronauta('André', estacion, consumo=8),
        Astronauta('Josefa', estacion, consumo=4),
        Astronauta('Daniel', estacion, consumo=5)
    ]

    for a in astronautas:
        a.start()
    for a in astronautas:
        a.join()

    print(f"Nivel final de oxígeno: {estacion.oxigeno}")
    print(f"Nivel esperado: 350")
    print(f"¿Correcto? {estacion.oxigeno == 350}\n")
    print(f"Registro final: {getattr(estacion, 'registro', 'No implementado')}")
    print(f"¿Registro correcto? {estacion.registro == {'Clemente': 500, 'Bastián': 300, 'André': 400, 'Josefa': 200, 'Daniel': 250}}")


## Ejercicio 2: Preparando el lanzamiento 🛸

Antes de que la nave pueda despegar, cada uno de sus módulos debe completar su verificación de forma **independiente** en paralelo. El controlador de lanzamiento debe esperar a que todos los módulos confirmen que están listos. Una vez que **todos** los módulos completaron la revisión, el Controlador autorizará el lanzamiento.

### Clases disponibles

#### Clase `Modulo`

Cada módulo es un hilo independiente. Atributos importantes:
- `self.nombre`: nombre del módulo.
- `self.tiempo_verificacion`: segundos que tarda la verificación (simula con `time.sleep`).
- `self.evento_listo`: un `threading.Event` que el módulo debe **activar** cuando termine su verificación.
- `self.evento_despegue`: un evento del controlador que avisará a los módulos cuando estén autorizados a iniciar.

#### Clase `ControladorLanzamiento`

Atributos importantes:
- `self.modulos`: lista de instancias de `Modulo`.
- `self.evento_despegue`: el evento global que se activará cuando **todos** los módulos estén listos para iniciar.

---

### Métodos a implementar

### 🔧 `Modulo.run(self)`

- Imprimir `🔧 [{nombre}] Iniciando verificación...`
- Simular la verificación con `time.sleep(self.tiempo_verificacion)`.
- Imprimir `✅ [{nombre}] Verificación completada.`
- Notificar al controlador que el módulo está listo.
- Esperar a que el controlador dé la orden de partida.
- Imprimir `🚀 [{nombre}] Módulo iniciado!`

### ⏳ `ControladorLanzamiento.esperar_confirmaciones(self)`

- Imprimir `⏳ Esperando confirmación de todos los módulos...`
- Para cada módulo en `self.modulos`:
  - Esperar a que el módulo señale que está listo.
  - Al confirmar, imprimir `📡 [{nombre}] Confirmado.`

### 🚀 `ControladorLanzamiento.ejecutar_lanzamiento(self)`

- Imprimir `🚀 ¡Todos los sistemas confirmados! Iniciando evento de lanzamiento global...`
- Dar la orden de partida a todos los módulos.
- Esperar a que todos los módulos se inicien`.
- Imprimir `🛑 Lanzamiento completado exitosamente.`


In [ ]:
import threading
import time
import random

random.seed('IIC2233')


class Modulo(threading.Thread):
    def __init__(self, nombre: str, tiempo_verificacion: float, evento_despegue: threading.Event) -> None:
        super().__init__(name=nombre)
        self.nombre = nombre
        self.tiempo_verificacion = tiempo_verificacion
        self.evento_listo = threading.Event()
        self.evento_despegue = evento_despegue

    def run(self) -> None:
        print(f"🔧 [{self.nombre}] Iniciando verificación...")
        time.sleep(self.tiempo_verificacion)
        print(f"✅ [{self.nombre}] Verificación completada.")
        self.evento_listo.set()  # Indica al modulo que el controlador está listo

        self.evento_despegue.wait()  # Espera orden del controlador para iniciar
        print(f"🚀 [{self.nombre}] Motores encendidos! Listos para la acción.")


class ControladorLanzamiento:
    def __init__(self) -> None:
        self.modulos: list[Modulo] = []
        self.evento_despegue = threading.Event()

    def agregar_modulo(self, nombre: str, tiempo_verificacion: float) -> None:
        self.modulos.append(Modulo(nombre, tiempo_verificacion, self.evento_despegue))

    def iniciar_verificaciones(self) -> None:
        print("🛰️  Iniciando verificaciones simultáneas de módulos...\n")
        for modulo in self.modulos:
            modulo.start()

    def esperar_confirmaciones(self) -> None:
        print("⏳ Esperando confirmación de todos los módulos...")
        for modulo in self.modulos:
            modulo.evento_listo.wait()  # Espera a que todos los controladores terminen la verificación
            print(f"📡 [{modulo.nombre}] Confirmado.")

    def ejecutar_lanzamiento(self) -> None:
        print("\n🚀 ¡Todos los sistemas confirmados! Iniciando evento de lanzamiento global...")
        self.evento_despegue.set()  # Indica a los módulos que pueden iniciar

        for modulo in self.modulos:
            modulo.join()

        print("\n🛑 Lanzamiento completado exitosamente.")


if __name__ == '__main__':

    controlador = ControladorLanzamiento()

    datos_modulos = [
        ('Motor Principal', random.uniform(1, 4)),
        ('Sistema de Navegación', random.uniform(1, 4)),
        ('Comunicaciones', random.uniform(1, 4)),
        ('Soporte Vital', random.uniform(1, 4)),
        ('Escudos Térmicos', random.uniform(1, 4)),
    ]

    for nombre, tiempo in datos_modulos:
        controlador.agregar_modulo(nombre, tiempo)

    controlador.iniciar_verificaciones()
    controlador.esperar_confirmaciones()
    controlador.ejecutar_lanzamiento()


## Preguntas 🤔

Analiza el siguiente código y determina qué se imprimirá en la consola al ejecutarlo. Como contiene threads _daemon_ que no se ejecutan correctamente en un notebook, ejecútalo desde el archivo `pregunta.py`.

```python
import threading
import time


def mision(nombre: str, duracion: float) -> None:
    time.sleep(duracion)
    print(f"🌟 {nombre}: misión completada")


t_rapido = threading.Thread(target=mision, args=("Corto", 1), daemon=True)
t_lento  = threading.Thread(target=mision, args=("Largo", 5), daemon=True)

t_rapido.start()
t_lento.start()

t_rapido.join()
print("Programa principal terminado")
```


A) Se imprime en orden: `🌟 Corto: misión completada`, `Programa principal terminado`, `🌟 Largo: misión completada`.

B) Se imprime en orden: `🌟 Corto: misión completada`, `Programa principal terminado`. El hilo `t_lento` **nunca imprime** porque es un *daemon* y el programa termina antes de que complete su tarea.

C) No se imprime nada porque ambos hilos son *daemon* y ninguno puede ejecutar `print()`.

D) Se produce un error porque no se puede llamar `join()` en un hilo *daemon*.

E) Se imprime `Programa principal terminado` antes que `🌟 Corto: misión completada` porque el hilo principal tiene prioridad.


---

✅ **Respuesta Correcta: B**

`t_rapido.join()` bloquea al hilo principal hasta que `t_rapido` termina (~1 segundo). En ese momento se imprime `🌟 Corto: misión completada` y luego `Programa principal terminado`.

Como no hay `t_lento.join()`, el hilo principal no tiene más instrucciones y **el programa termina**. Al terminar el único hilo no-*daemon*, Python interrumpe todos los hilos *daemon* que aún estén corriendo, sin importar en qué punto estén. Por eso `t_lento` nunca llega a imprimir su mensaje.




---
Analiza el siguiente código y determina qué se va a imprimir al ejecutarlo.


In [ ]:
import threading
import time

def cruzar(auto, indice, auto_siguiente, print_lock):
    with print_lock:
        print(f"🚦 auto {indice} esperando pasar!")
    time.sleep(1)
    if indice == 2:
        auto_siguiente.wait(timeout=0)
    else:
        auto_siguiente.wait()
        time.sleep(1)
    with print_lock:
        print(f"🚗 auto {indice} pasando...")
    auto.set()

print_lock = threading.Lock()

autos = []
for _ in range(4):
    autos.append(threading.Event())

threads = []
for i in range(4):
    thread = threading.Thread(target=cruzar, args=(autos[i], i, autos[i - 1], print_lock))
    threads.append(thread)
    thread.start()


A) Ninguno de los autos logra pasar ya que hay un _deadlock_.

B) Sólo el auto 2 logra pasar.

C) Los cuatro autos logran pasar, en el orden 0, 1, 2, 3.

D) Los cuatro autos logran pasar, en el orden 2, 3, 0, 1.

E) Se levanta un `IndexError` al acceder a `autos[i - 1]`.

---
✅ **Respuesta Correcta: D**

Los cuatro autos imprimen `"🚦 auto {indice} esperando pasar!"` y luego esperan un segundo. Cada uno de los autos espera al auto anterior (con el auto cero esperando al auto tres), pero como el _timeout_ para el auto dos es igual a cero, éste logra pasar, por lo que luego de él pasa el auto tres que lo estaba esperando, luego el auto cero que esperaba al auto tres, y finalmente el auto uno que esperaba al auto cero.

---
Analiza el siguiente código e indica qué se imprime al correrlo:

In [ ]:
import threading

def f_thread_1(lock):
    lock.acquire()
    print("1", end=" ")

def f_thread_2(evento, lock):
    evento.wait()
    lock.acquire()
    print("2", end=" ")

def f_thread_3(evento):
    evento.set()
    print("3", end=" ")

evento = threading.Event()
lock = threading.Lock()

thread_1 = threading.Thread(target=f_thread_1, args=(lock,))
thread_2 = threading.Thread(target=f_thread_2, args=(evento, lock))
thread_3 = threading.Thread(target=f_thread_3, args=(evento,))

thread_2.start()
thread_1.start()
thread_1.join()
thread_3.start()

A) 1 3

B) 2 3

C) 1 3 2

D) 2 3 1

E) No se imprime nada

---

✅ **Respuesta Correcta: A**

Primero empieza el _thread_ dos y se queda esperando el evento. Luego empieza el _thread_ uno, adquiere el _lock_ ya que no lo tiene nadie e imprime `"1"` y termina. Luego hacemos `join` al _thread_ uno, y cuando termina, empieza el _thread_ tres, que setea el evento e imprime `"3"`. Esto hace que el _thread_ dos resuma su ejecución, pero luego pide adquirir el _lock_, y como el _thread_ uno nunca lo soltó, se queda pegado esperando y no imprime nada.